# 01 — Ingest & Transform

Loads the raw synthetic finance-ops CSVs (`data/raw/`) and produces a cleaned, analysis-ready table in `data/processed/`.

**Scope of this notebook:** standardize schema and fix data-quality mess (inconsistent text casing/whitespace, mixed date-string formats, a handful of missing timestamps). It does **not** compute KPIs or detect anomalies — that's `src/kpis.py` and `src/anomalies.py` (stage 3), which read the *cleaned* output this notebook writes.

**Local vs. Databricks:** the same notebook runs unmodified in both places. The first code cell detects Databricks and switches paths: locally it reads/writes `data/raw` and `data/processed`; on Databricks (Free Edition) it reads the CSVs from a Unity Catalog **Volume** (`/Volumes/workspace/finance_ops/files/raw`). All cleaning is plain pandas either way — at ~2,500 rows, distributed processing isn't justified.

The one Databricks-only addition is at the end: the cleaned pandas DataFrames are registered as **Delta tables** in Unity Catalog (`workspace.finance_ops.*`) and cross-checked with a Databricks SQL query. Spark is used there only as the write/query interface to the catalog — no transformation logic runs in Spark.

In [1]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Databricks sets this env var on every cluster/serverless runtime.
ON_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ

# Unity Catalog location (Databricks only): catalog.schema, plus a Volume for files.
CATALOG_SCHEMA = 'workspace.finance_ops'
VOLUME_DIR = Path('/Volumes/workspace/finance_ops/files')

if ON_DATABRICKS:
    RAW_DIR = VOLUME_DIR / 'raw'
    PROCESSED_DIR = VOLUME_DIR / 'processed'
else:
    RAW_DIR = Path('..') / 'data' / 'raw'
    PROCESSED_DIR = Path('..') / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'running on Databricks: {ON_DATABRICKS} | pandas {pd.__version__} | raw data from {RAW_DIR}')
pd.set_option('display.max_columns', None)

running on Databricks: False | pandas 2.2.0 | raw data from ..\data\raw


In [2]:
invoices_raw = pd.read_csv(RAW_DIR / 'invoices.csv', dtype=str)
vendors_raw = pd.read_csv(RAW_DIR / 'vendors.csv', dtype=str)
budgets = pd.read_csv(RAW_DIR / 'budgets.csv')

print(f'invoices: {len(invoices_raw)} rows')
print(f'vendors:  {len(vendors_raw)} rows')
print(f'budgets:  {len(budgets)} rows')
invoices_raw.head(3)

invoices: 2550 rows
vendors:  40 rows
budgets:  432 rows


,invoice_id,vendor_id,department,category,invoice_date,submitted_date,approved_date,paid_date,amount,currency,status
0,INV00001,V020,Legal,Software & IT Services,2026-01-19,2026-01-23,2026-01-26,2026-01-29,2122.75,EUR,Paid
1,INV00002,V003,Operations,Consulting,2026-07-17,2026-07-18,2026-07-22,2026-07-23,760.84,EUR,Paid
2,INV00003,V016,Marketing,Travel,2026-04-13,2026-04-13,2026-04-21,2026-04-26,19385.8,EUR,Paid


## Explore the mess

Before cleaning anything, look at what's actually wrong. This is the step that turns "clean this data" from a guess into a checklist.

In [3]:
print('department value counts:')
print(invoices_raw['department'].value_counts())
print()
print('category value counts:')
print(invoices_raw['category'].value_counts())
print()
print('status value counts:')
print(invoices_raw['status'].value_counts())
print()

iso_pattern = re.compile(r'^\d{4}-\d{2}-\d{2}$')
non_iso_dates = (~invoices_raw['invoice_date'].str.match(iso_pattern)).sum()
print(f'invoice_date values not in ISO format: {non_iso_dates}')
print(f'missing submitted_date: {invoices_raw["submitted_date"].isna().sum()}')

department value counts:
department
Operations        398
Legal             391
HR                368
IT                364
Finance           362
Marketing         362
  Legal            27
Hr                 23
MARKETING          21
finance            20
marketing          18
OPERATIONS         18
It                 18
it                 18
  Marketing        17
legal              17
LEGAL              16
FINANCE            15
operations         15
  HR               15
  Finance          13
  IT               12
hr                 12
  Operations       10
Name: count, dtype: int64

category value counts:
category
Professional Services         506
Consulting                    495
Software & IT Services        452
Office Supplies               374
Travel                        244
Facilities                    179
  Professional Services        27
  Consulting                   27
PROFESSIONAL SERVICES          23
SOFTWARE & IT SERVICES         23
  Software & IT Services       23
CON

## Standardize categorical text

`department`/`category`/`status` carry the same value in different casings and with stray whitespace (`" it "`, `"IT"`, `"It"`). Blindly calling `.str.title()` would break an abbreviation like `"IT"` (title-cases to `"It"`). Instead we **casefold + strip, then map through a lookup table** built from the known canonical values — a more robust pattern for any fixed-vocabulary field, not just this dataset.

In [4]:
DEPARTMENTS = ['Finance', 'IT', 'Marketing', 'Operations', 'HR', 'Legal']
CATEGORIES = ['Consulting', 'Software & IT Services', 'Facilities', 'Travel', 'Office Supplies', 'Professional Services']
STATUSES = ['Paid']

def build_canon_map(canonical_values):
    return {v.strip().lower(): v for v in canonical_values}

def normalize_categorical(series: pd.Series, canon_map: dict) -> pd.Series:
    cleaned = series.str.strip().str.lower().map(canon_map)
    unmatched = cleaned.isna().sum()
    if unmatched:
        print(f'  warning: {unmatched} values in {series.name!r} did not match the canonical map')
    return cleaned

invoices = invoices_raw.copy()
invoices['department'] = normalize_categorical(invoices['department'], build_canon_map(DEPARTMENTS))
invoices['category'] = normalize_categorical(invoices['category'], build_canon_map(CATEGORIES))
invoices['status'] = normalize_categorical(invoices['status'], build_canon_map(STATUSES))

assert invoices['department'].isin(DEPARTMENTS).all()
assert invoices['category'].isin(CATEGORIES).all()
assert invoices['status'].isin(STATUSES).all()
invoices[['department', 'category', 'status']].nunique()

department    6
category      6
status        1
dtype: int64

## Clean vendor names

`vendor_name` is free text (not a fixed vocabulary), so there's no lookup table — just strip leading/trailing whitespace and collapse repeated internal spaces.

In [5]:
vendors = vendors_raw.copy()
vendors['vendor_name'] = vendors['vendor_name'].str.strip().str.replace(r'\s+', ' ', regex=True)
vendors.head(3)

,vendor_id,vendor_name,category
0,V001,"Rodriguez, Figueroa and Sanchez",Professional Services
1,V002,Doyle Ltd,Consulting
2,V003,"Mcclain, Miller and Henderson",Consulting


## Parse mixed date formats

`invoice_date` shows up in three different string formats (`YYYY-MM-DD`, `MM/DD/YYYY`, `DD-Mon-YYYY`, `YYYY/MM/DD`) — a realistic symptom of data entered by different people/systems. `pandas.to_datetime` can't infer a single consistent format across all of them reliably, so we try a list of known formats per value instead of relying on format inference.

In [6]:
KNOWN_DATE_FORMATS = ['%Y-%m-%d', '%m/%d/%Y', '%d-%b-%Y', '%Y/%m/%d']

def parse_mixed_dates(series: pd.Series) -> pd.Series:
    parsed = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns]')
    remaining = series.notna()
    for fmt in KNOWN_DATE_FORMATS:
        if not remaining.any():
            break
        attempt = pd.to_datetime(series[remaining], format=fmt, errors='coerce')
        matched = attempt.notna()
        parsed.loc[attempt.index[matched]] = attempt[matched]
        remaining.loc[attempt.index[matched]] = False
    return parsed

invoices['invoice_date'] = parse_mixed_dates(invoices['invoice_date'])
invoices['submitted_date'] = parse_mixed_dates(invoices['submitted_date'])
invoices['approved_date'] = parse_mixed_dates(invoices['approved_date'])
invoices['paid_date'] = parse_mixed_dates(invoices['paid_date'])

unparsed = invoices['invoice_date'].isna().sum()
print(f'invoice_date rows that failed to parse: {unparsed} (should be 0)')
invoices[['invoice_date', 'submitted_date', 'approved_date', 'paid_date']].dtypes

invoice_date rows that failed to parse: 0 (should be 0)


invoice_date      datetime64[ns]
submitted_date    datetime64[ns]
approved_date     datetime64[ns]
paid_date         datetime64[ns]
dtype: object

## Handle missing `submitted_date`

A few rows (~1%) are missing `submitted_date` entirely — a plausible real-world gap (a workflow step that didn't log a timestamp). We don't drop these rows outright: `amount`/`department`/`category` are still valid, so they should still count toward spend KPIs. We only flag them so the processing-time KPI (which needs `submitted_date`) can exclude them explicitly instead of silently mis-computing on a NaT.

In [7]:
invoices['missing_submitted_date'] = invoices['submitted_date'].isna()
invoices['processing_days'] = (invoices['paid_date'] - invoices['submitted_date']).dt.days

invoices['amount'] = pd.to_numeric(invoices['amount'])

print(f"flagged missing_submitted_date: {invoices['missing_submitted_date'].sum()} rows")
print(f"processing_days is null for: {invoices['processing_days'].isna().sum()} rows (should match the flagged count)")

flagged missing_submitted_date: 24 rows
processing_days is null for: 24 rows (should match the flagged count)


## Validate before writing

Cheap assertions that would catch a broken cleaning step before it silently corrupts stage 3 (KPIs/anomaly detection).

In [8]:
assert invoices['invoice_id'].is_unique, 'invoice_id must stay unique — planted duplicates use a new id, not a repeated one'
assert len(invoices) == len(invoices_raw), 'row count must be preserved — this notebook cleans, it does not drop/dedupe'
assert invoices['amount'].gt(0).all(), 'amount must stay positive after cleaning'
assert invoices['department'].isin(DEPARTMENTS).all()
assert invoices['category'].isin(CATEGORIES).all()

print('all validation checks passed')

all validation checks passed


## Join clean vendor names and write output

Attach the cleaned `vendor_name` for display purposes (KPI/anomaly logic keys off `vendor_id`, never the name). `budgets.csv` was never messified at generation time, so it's copied through unchanged for consistency — stage 3 should only ever read from `data/processed/`.

In [9]:
invoices_clean = invoices.merge(vendors[['vendor_id', 'vendor_name']], on='vendor_id', how='left')

invoices_clean.to_parquet(PROCESSED_DIR / 'invoices_clean.parquet', index=False)
vendors.to_parquet(PROCESSED_DIR / 'vendors_clean.parquet', index=False)
budgets.to_parquet(PROCESSED_DIR / 'budgets.parquet', index=False)

print(f'wrote {len(invoices_clean)} cleaned invoice rows to {PROCESSED_DIR}')
invoices_clean.head(3)

wrote 2550 cleaned invoice rows to ..\data\processed


,invoice_id,vendor_id,department,category,invoice_date,submitted_date,approved_date,paid_date,amount,currency,status,missing_submitted_date,processing_days,vendor_name
0,INV00001,V020,Legal,Software & IT Services,2026-01-19,2026-01-23,2026-01-26,2026-01-29,2122.75,EUR,Paid,False,6.0,"Reid, Ferguson and Sanchez"
1,INV00002,V003,Operations,Consulting,2026-07-17,2026-07-18,2026-07-22,2026-07-23,760.84,EUR,Paid,False,5.0,"Mcclain, Miller and Henderson"
2,INV00003,V016,Marketing,Travel,2026-04-13,2026-04-13,2026-04-21,2026-04-26,19385.80,EUR,Paid,False,13.0,Galloway-Wyatt


## Databricks only: register Delta tables in Unity Catalog

Parquet files in a folder are just files; a **Delta table** in Unity Catalog is a named, governed table (`workspace.finance_ops.invoices_clean`) that anyone in the workspace can find in the Catalog Explorer and query with SQL — with ACID writes and version history ("time travel") on top of Parquet.

`spark.createDataFrame(pandas_df)` is used purely to hand the already-cleaned pandas data to the catalog. Skipped automatically when running locally.

In [10]:
if ON_DATABRICKS:
    tables = {
        'invoices_clean': invoices_clean,
        'vendors_clean': vendors,
        'budgets': budgets,
    }
    for table_name, df in tables.items():
        (spark.createDataFrame(df)
            .write.mode('overwrite')
            .option('overwriteSchema', 'true')
            .saveAsTable(f'{CATALOG_SCHEMA}.{table_name}'))
        print(f'wrote Delta table {CATALOG_SCHEMA}.{table_name} ({len(df)} rows)')
else:
    print('local run: skipping Delta table registration (parquet output above is all stage 3 needs)')

local run: skipping Delta table registration (parquet output above is all stage 3 needs)


## Databricks only: cross-check with Databricks SQL

Query the Delta table with SQL and assert it agrees with pandas — spend per department must be identical whichever engine computes it. This is a validation step, not part of the pipeline: stage 3 (`src/kpis.py`) still reads the local parquet output.

In [11]:
if ON_DATABRICKS:
    spend_sql = spark.sql(f'''
        SELECT department,
               COUNT(*)              AS invoices,
               ROUND(SUM(amount), 2) AS total_spend
        FROM {CATALOG_SCHEMA}.invoices_clean
        GROUP BY department
        ORDER BY total_spend DESC
    ''')
    display(spend_sql)

    sql_totals = spend_sql.toPandas().set_index('department')['total_spend'].sort_index()
    pandas_totals = invoices_clean.groupby('department')['amount'].sum().round(2).sort_index()
    assert (sql_totals - pandas_totals).abs().max() < 0.01, 'Databricks SQL and pandas disagree'
    print('Databricks SQL totals match pandas for every department')
else:
    print('local run: skipping Databricks SQL cross-check')

local run: skipping Databricks SQL cross-check


## Cleaning summary

For the README's honest-limitations section.

In [12]:
print('Cleaning summary')
print('-----------------')
print(f"rows in / out:              {len(invoices_raw)} / {len(invoices_clean)}")
print(f"invoice_date reparsed:      {non_iso_dates} rows had a non-ISO source format")
print(f"rows missing submitted_date:{int(invoices['missing_submitted_date'].sum())} (kept, excluded from processing-time KPI only)")
print(f"vendor names whitespace-fixed: {(vendors_raw['vendor_name'] != vendors['vendor_name']).sum()}")

Cleaning summary
-----------------
rows in / out:              2550 / 2550
invoice_date reparsed:      202 rows had a non-ISO source format
rows missing submitted_date:24 (kept, excluded from processing-time KPI only)
vendor names whitespace-fixed: 3
